# 注册算子

Relay是TVM的高级IR，支持深度学习模型的表示、优化和执行。Relay的算子系统包括：
- 算子元信息（名称、参数、属性等）
- 类型关系（输入输出类型约束）
- 计算函数（实现具体计算逻辑）
- 策略函数（选择最佳实现）

In [1]:
import set_env

In [2]:
import tvm

def register(op_name, describe=""):
    """Get the Op for a given name.
    when the op_name is not registered, create a new empty op with the given name.
    when the op_name has been registered, abort with an error message.

    Parameters
    ----------
    op_name : str
        The operator name

    describe : Optional[str]
        The operator description
    """

    tvm.ir._ffi_api.RegisterOp(op_name, describe)

## 1. 获取或创建算子

In [3]:
op_name = "my.operator"
try:
    op = tvm.ir.Op.get(op_name)
except:
    op = register(op_name)

## 2. 设置算子属性

In [4]:
tvm.ir.Op.get(op_name).set_num_inputs(1)
tvm.ir.Op.get(op_name).add_argument("data", "Tensor", "输入数据")

## 3. 定义类型关系
类型关系函数确保输入输出类型的一致性：

In [5]:
def my_op_type_rel(arg_types, attrs):
    inputa_type = arg_types[0]
    # print(inputa_type)
    return tvm.relay.TensorType(inputa_type.shape, inputa_type.dtype)
tvm.ir.Op.get(op_name).add_type_rel("MyOpTypeRel", my_op_type_rel)

## 4. 设置计算函数

计算函数实现具体的计算逻辑：
```python
@tvm.te.tag_scope(op_name)
@tvm.relay.op.register_compute(op_name)
def my_op_compute(attrs, inputs, output_type):
    x = inputs[0]
    out = x + 2 * 1
    return [out]
```

或者：

In [6]:
def my_op_compute(attrs, inputs, output_type):
    x = inputs[0]
    out = x + 2 * 1
    return [out]

tvm.relay.op.register_compute(op_name, my_op_compute, level=10)

<function __main__.my_op_compute(attrs, inputs, output_type)>

## 5. 设置算子模式

In [7]:
tvm.ir.Op.get(op_name).set_attr("TOpPattern", tvm.relay.op.OpPattern.ELEMWISE)

## 6. 提供 Python API 接口

为了方便使用，需要提供用户友好的Python API：

In [8]:
def my_operator(data):
    """用户友好的API接口"""
    op = tvm.ir.Op.get(op_name)
    # attrs = tvm.ir.make_node("DictAttrs")
    return tvm.relay.Call(op, [data], attrs=None, type_args=None)

## 7. 实现算子调度

为不同目标设备提供调度的实现策略：

In [9]:
@tvm.target.generic_func
def schedule_special_op(attrs, outs, target):
    with target:
        outs = [outs] if isinstance(outs, tvm.te.tensor.Tensor) else outs
        output = outs[0]
        s = tvm.te.create_schedule(output.op)   
        return s
tvm.relay.op.op.register_schedule(op_name, schedule_special_op)

GenericFunc(0x563dd948ff40)

## 8. 测试与验证

创建全面的测试用例确保算子功能正确：

In [10]:
import numpy as np
# 创建测试数据
np_data = np.random.randn(10).astype("float32")

# 创建Relay函数
x = tvm.relay.var("x", shape=(10,), dtype="float32")
y = my_operator(x)
func = tvm.relay.Function([x], y)
mod = tvm.IRModule.from_expr(func)

# 执行计算
intrp = tvm.relay.create_executor(kind="debug", mod=mod, device=tvm.cpu(0))
result = intrp.evaluate()(np_data)

# 验证结果
expected = np_data + 2 * 1   # 假设这是算子的预期行为
np.testing.assert_allclose(result.numpy(), expected, rtol=1e-5)
print("测试通过!")

测试通过!


In [11]:
mod.show()

## 创建统一接口

In [12]:
from dataclasses import dataclass, field, asdict
import tvm

def register_op(op_name, describe=""):
    """Get the Op for a given name.
    when the op_name is not registered, create a new empty op with the given name.
    when the op_name has been registered, abort with an error message.

    Parameters
    ----------
    op_name : str
        The operator name

    describe : Optional[str]
        The operator description
    """

    return tvm.ir._ffi_api.RegisterOp(op_name, describe)

@tvm.target.generic_func
def schedule_special_op(attrs, outs, target):
    with target:
        outs = [outs] if isinstance(outs, tvm.te.tensor.Tensor) else outs
        output = outs[0]
        s = tvm.te.create_schedule(output.op)   
        return s

@dataclass
class RegisterOp:
    op_name: str
    op: tvm.ir.op.Op = field(init=False)

    def __post_init__(self):
        try:
            self.op = tvm.ir.Op.get(self.op_name)
        except:
            register_op(self.op_name)
            self.op = tvm.ir.Op.get(self.op_name)

    def add_type_rel(self, rel_name, type_rel_func=None):
        self.op.add_type_rel(rel_name, type_rel_func)

    def register_compute(self, compute=None, level=10):
        return tvm.relay.op.register_compute(self.op_name, compute, level=level)

    def register_schedule(self):
        return tvm.relay.op.op.register_schedule(self.op_name, schedule_special_op)

    def register_pattern(self, pattern, level=10):
        """Register operator pattern for an op.

        Parameters
        ----------
        op_name : str
            The name of the op.

        pattern : int
            The pattern being used.

        level : int
            The priority level
        """
        return tvm.ir.register_op_attr(self.op_name, "TOpPattern", pattern, level)

    def __call__(self, args, attrs=None, type_args=None, span=None):
        """用户友好的回调 API 接口"""
        return tvm.relay.Call(self.op, args, attrs=attrs, type_args=type_args, span=span)

    def register(self, pattern, rel_name, type_rel_func=None, compute=None, level=10):
        self.add_type_rel(rel_name, type_rel_func)
        self.register_compute(compute, level=level)
        self.register_schedule()
        self.register_pattern(pattern, level=level)

if __name__ == "__main__":
    import numpy as np
    # 创建测试数据
    np_data = np.random.randn(10).astype("float32")
    op_name = "my.operator2"
    def my_op_type_rel(arg_types, attrs):
        inputa_type = arg_types[0]
        return tvm.relay.TensorType(inputa_type.shape, inputa_type.dtype)
    def my_op_compute(attrs, inputs, output_type):
        x = inputs[0]
        out = x + 2 * 1
        return [out]
    op = RegisterOp(op_name)
    pattern = tvm.relay.op.OpPattern.ELEMWISE
    rel_name = "MyOpTypeRel"
    type_rel_func = my_op_type_rel
    compute = my_op_compute
    op.register(pattern, rel_name, type_rel_func, compute, level=10)
    x = tvm.relay.var("x", shape=(10,), dtype="float32")
    y = op([x])
    mod = tvm.IRModule.from_expr(tvm.relay.Function([x], y))
    # 执行计算
    intrp = tvm.relay.create_executor(kind="vm", mod=mod, device=tvm.cpu(0))
    result = intrp.evaluate()(np_data)
    # 验证结果
    expected = np_data + 2 * 1   # 假设这是算子的预期行为
    np.testing.assert_allclose(result.numpy(), expected, rtol=1e-5)
    print(f"{op} 测试通过!")
    mod.show()

RegisterOp(op_name='my.operator2', op=Op(my.operator2)) 测试通过!
